# VGGT-SLAM GPU Reconstruction
Runtime → GPU → Run all → Upload images → Download

## 1. Check GPU

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Install Dependencies

In [ ]:
!apt-get update -qq && apt-get install -y -qq git libboost-all-dev cmake gcc g++ > /dev/null 2>&1
print('✅ Done')

## 3. Setup VGGT-SLAM

In [ ]:
import os
import shutil

# Clean up any existing installation to ensure fresh setup
if os.path.exists('VGGT-SLAM'):
    print('🧹 Removing existing VGGT-SLAM directory...')
    shutil.rmtree('VGGT-SLAM')

!git clone https://github.com/MIT-SPARK/VGGT-SLAM.git
print('✅ Cloned VGGT-SLAM')

In [ ]:
%%bash
cd VGGT-SLAM
chmod +x setup.sh
# Run setup script (installs gtsam, salad, viser)
./setup.sh
echo "✅ Setup complete"

## 4. Upload Images

In [ ]:
from google.colab import files
import zipfile, os, shutil
from PIL import Image

if os.path.exists('input_images'):
    shutil.rmtree('input_images')
os.makedirs('input_images')

print('📤 Upload images_for_vggt_slam.zip')
uploaded = files.upload()

for f in uploaded:
    if f.endswith('.zip'):
        print(f'Extracting {f}...')
        with zipfile.ZipFile(f) as z:
            z.extractall('input_images')
        os.remove(f)

imgs = [f for f in os.listdir('input_images') if f.lower().endswith(('.jpg','.png'))]
print(f'✅ {len(imgs)} images')

# VGGT-SLAM requires EXACT pixel dimensions (not just same ratio)
# To avoid OOM, downscale to 960x540 (16:9, safe for model)
print('🔧 Normalizing image sizes...')
target_size = (960, 540)
resized_count = 0
for img_file in imgs:
    path = os.path.join('input_images', img_file)
    with Image.open(path) as img:
        if img.size != target_size:
            img = img.resize(target_size, Image.LANCZOS)
            img.save(path, quality=95)
            resized_count += 1
print(f'✅ All images normalized to 960×540 ({resized_count} resized)')

## 4.5 Pre-cache SALAD Model (Fixes GitHub timeout)

In [ ]:
import os
import torch

# Pre-download models to torch hub cache to avoid GitHub API timeout
hub_dir = torch.hub.get_dir()

# 1. SALAD model
salad_dir = os.path.join(hub_dir, 'serizba_salad_main')
if not os.path.exists(salad_dir):
    print(f'📥 Pre-caching SALAD model...')
    !git clone https://github.com/serizba/salad.git "$salad_dir"
    print('✅ SALAD cached')
else:
    print(f'✅ SALAD already cached')

# 2. DINOv2 model (required by SALAD)
dinov2_dir = os.path.join(hub_dir, 'facebookresearch_dinov2_main')
if not os.path.exists(dinov2_dir):
    print(f'📥 Pre-caching DINOv2 model...')
    !git clone https://github.com/facebookresearch/dinov2.git "$dinov2_dir"
    print('✅ DINOv2 cached')
else:
    print(f'✅ DINOv2 already cached')

print('\n✅ All models pre-cached!')

## 5. Run VGGT-SLAM (30-40 min)

In [ ]:
import torch
import gc

# Clear GPU cache to prevent OOM from previous runs
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print('✅ GPU cache cleared')

# Run VGGT-SLAM
get_ipython().run_cell_magic('bash', '', '''cd VGGT-SLAM
mkdir -p ../vggt_logs
echo '🚀 Starting VGGT-SLAM...'
python3 main.py \\
  --image_folder ../input_images \\
  --max_loops 3 \\
  --conf_threshold 25.0 \\
  --log_results \\
  --log_path ../vggt_logs/poses.txt
echo '✅ Done'
''')

## 6. Export Point Cloud

In [ ]:
import glob, os, shutil
import numpy as np
from PIL import Image

print('📊 Checking output...\n')

# Check vggt_logs
if not os.path.exists('vggt_logs'):
    print('❌ vggt_logs does not exist!')
    raise Exception('VGGT-SLAM did not run')

print('Files in vggt_logs:')
for item in os.listdir('vggt_logs'):
    if os.path.isdir(os.path.join('vggt_logs', item)):
        print(f'  {item}/ (folder)')
    else:
        print(f'  {item}')
print()

# Get input images
image_files = sorted([f for f in os.listdir('input_images') 
                      if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
print(f'Found {len(image_files)} input images\n')

# Find point clouds
npz_files = glob.glob('vggt_logs/poses_logs/*.npz')
print(f'Found {len(npz_files)} NPZ files\n')

if not npz_files:
    print('❌ NO POINT CLOUDS!')
    raise Exception('No NPZ files found in poses_logs/')

os.makedirs('vggt_output', exist_ok=True)

# Process NPZ files
print(f'Processing {len(npz_files)} NPZ files with colors...\n')
all_pts, all_cols = [], []

for npz_path in sorted(npz_files):
    fname = os.path.basename(npz_path)
    frame_idx = int(fname.split('.')[0])  # Extract frame number from filename
    
    print(f'  Reading {fname}...')
    data = np.load(npz_path)
    
    # Load corresponding image
    if frame_idx < len(image_files):
        img_path = os.path.join('input_images', image_files[frame_idx])
        img = Image.open(img_path)
        img_array = np.array(img)
        
        # Resize image to match point cloud dimensions if needed
        pc = data['pointcloud']
        h, w = pc.shape[:2]
        
        if img_array.shape[:2] != (h, w):
            img = img.resize((w, h), Image.LANCZOS)
            img_array = np.array(img)
        
        if len(all_pts) == 0:
            print(f'    Point cloud shape: {pc.shape}')
            print(f'    Image shape: {img_array.shape}')
        
        # Reshape point cloud from (H, W, 3) to (H*W, 3)
        if pc.ndim == 3 and pc.shape[2] == 3:
            pts = pc.reshape(-1, 3)
            
            # Reshape colors to match
            if img_array.ndim == 3 and img_array.shape[2] >= 3:
                cols = img_array[:, :, :3].reshape(-1, 3).astype(np.uint8)
            else:
                # Grayscale image
                gray = img_array.reshape(-1, 1)
                cols = np.repeat(gray, 3, axis=1).astype(np.uint8)
            
            # Apply mask if available
            mask = data.get('mask', None)
            if mask is not None:
                mask_flat = mask.reshape(-1)
                pts = pts[mask_flat]
                cols = cols[mask_flat]
            
            # Filter out invalid points
            valid = np.isfinite(pts).all(axis=1) & (np.abs(pts).sum(axis=1) > 0.01)
            pts = pts[valid]
            cols = cols[valid]
            
            if len(pts) > 0:
                all_pts.append(pts)
                all_cols.append(cols)
                print(f'    ✓ {len(pts):,} colored points')
            else:
                print(f'    ⚠ No valid points after filtering')
    else:
        print(f'    ⚠ No matching image found')

if not all_pts:
    print('\n❌ No valid point data found!')
    raise Exception('Could not extract points from NPZ files')

# Merge all points
pts = np.vstack(all_pts)
cols = np.vstack(all_cols)

print(f'\n✅ Merged {len(pts):,} colored points\n')

# Write PLY
ply = 'vggt_output/point_cloud.ply'
with open(ply, 'w') as f:
    f.write('ply\n')
    f.write('format ascii 1.0\n')
    f.write(f'element vertex {len(pts)}\n')
    f.write('property float x\n')
    f.write('property float y\n')
    f.write('property float z\n')
    f.write('property uchar red\n')
    f.write('property uchar green\n')
    f.write('property uchar blue\n')
    f.write('end_header\n')
    for (x,y,z), (r,g,b) in zip(pts, cols):
        f.write(f'{x} {y} {z} {int(r)} {int(g)} {int(b)}\n')

print(f'✅ point_cloud.ply ({os.path.getsize(ply)/1024/1024:.1f} MB)')

# Copy poses
if os.path.exists('vggt_logs/poses.txt'):
    shutil.copy('vggt_logs/poses.txt', 'vggt_output/camera_poses.txt')
    print('✅ camera_poses.txt')

print('\n✅ Export complete!')

## 7. Download

In [ ]:
import zipfile, os
from google.colab import files

print('📦 Creating ZIP...\n')

if not os.path.exists('vggt_output'):
    print('❌ vggt_output does not exist!')
    raise Exception('No output')

output_files = []
for root, dirs, fnames in os.walk('vggt_output'):
    for f in fnames:
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp)
        output_files.append((fp, f))
        print(f'  {f} ({sz/1024:.1f} KB)')

if not output_files:
    print('\n❌ vggt_output is EMPTY!')
    raise Exception('No files to zip')

with zipfile.ZipFile('vggt_slam_output.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for fp, fn in output_files:
        z.write(fp, fn)

sz = os.path.getsize('vggt_slam_output.zip')/1024/1024
print(f'\n✅ ZIP: {sz:.1f} MB')
print('📥 Downloading...')
files.download('vggt_slam_output.zip')
print('✅ DONE')